In [1]:
# CELL 1: Sanity check – where are we running from?

import os

BASE_DIR = os.getcwd()

print("Current working directory:")
print(BASE_DIR)

print("\nContents of BASE_DIR:")
print(os.listdir(BASE_DIR))

Current working directory:
/mnt/e/SEM6/VR/MiniProject1/Dataset

Contents of BASE_DIR:
['json_for_validation', 'json_for_validation.zip', 'prune_dataset.ipynb', 'test', 'test.zip', 'train', 'train.zip', 'validation', 'validation.zip']


In [2]:
# CELL 2: Count category frequencies from TRAIN split

import os
import json
import glob
from collections import Counter

# Path to TRAIN annotations
TRAIN_ANNO_DIR = os.path.join(BASE_DIR, "train", "train", "annos")

print("Reading annotations from:", TRAIN_ANNO_DIR)

counter = Counter()

# Loop through all train annotation files
for ann_file in glob.glob(os.path.join(TRAIN_ANNO_DIR, "*.json")):
    with open(ann_file, "r") as f:
        data = json.load(f)

    for k, v in data.items():
        if k.startswith("item"):
            counter[v["category_id"]] += 1

# Show full category frequency (optional but useful)
print("\nCategory frequencies (category_id : count):")
for cid, cnt in counter.most_common():
    print(cid, ":", cnt)

# Get top-5 categories
top5 = [cid for cid, _ in counter.most_common(5)]

print("\nTop-5 category IDs:", top5)

Reading annotations from: /mnt/e/SEM6/VR/MiniProject1/Dataset/train/train/annos

Category frequencies (category_id : count):
1 : 71645
8 : 55387
7 : 36616
2 : 36064
9 : 30835
12 : 17949
10 : 17211
5 : 16095
4 : 13457
11 : 7907
13 : 6492
6 : 1985
3 : 543

Top-5 category IDs: [1, 8, 7, 2, 9]


In [4]:
# CELL 3: Prune TRAIN split (ROBUST VERSION)

import os
import json
import glob
import shutil

TRAIN_IMG_DIR = os.path.join(BASE_DIR, "train", "train", "image")
TRAIN_ANNO_DIR = os.path.join(BASE_DIR, "train", "train", "annos")

OUT_TRAIN_IMG_DIR = os.path.join(BASE_DIR, "DeepFashion2_pruned", "train", "images")
OUT_TRAIN_ANNO_DIR = os.path.join(BASE_DIR, "DeepFashion2_pruned", "train", "annos")

os.makedirs(OUT_TRAIN_IMG_DIR, exist_ok=True)
os.makedirs(OUT_TRAIN_ANNO_DIR, exist_ok=True)

print("Pruning TRAIN split...")
print("Top-5 categories used:", top5)

kept_images = 0
total_images = 0

for idx, ann_file in enumerate(glob.glob(os.path.join(TRAIN_ANNO_DIR, "*.json"))):
    total_images += 1

    if idx % 5000 == 0:
        print(f"Processed {idx} annotations...")

    img_id = os.path.basename(ann_file).replace(".json", "")
    img_path = os.path.join(TRAIN_IMG_DIR, img_id + ".jpg")

    if not os.path.exists(img_path):
        continue

    out_img_path = os.path.join(OUT_TRAIN_IMG_DIR, img_id + ".jpg")
    out_anno_path = os.path.join(OUT_TRAIN_ANNO_DIR, img_id + ".json")

    # skip if already processed
    if os.path.exists(out_img_path) and os.path.exists(out_anno_path):
        continue

    with open(ann_file, "r") as f:
        data = json.load(f)

    items = []
    for k, v in data.items():
        if k.startswith("item") and v["category_id"] in top5:
            items.append({
                "category_id": v["category_id"],
                "bbox": v["bounding_box"],
                "segmentation": v["segmentation"]
            })

    if not items:
        continue

    shutil.copy(img_path, out_img_path)

    with open(out_anno_path, "w") as f:
        json.dump({"items": items}, f)

    kept_images += 1

print("Total TRAIN images scanned:", total_images)
print("TRAIN images kept after pruning:", kept_images)

Pruning TRAIN split...
Top-5 categories used: [1, 8, 7, 2, 9]
Processed 0 annotations...
Processed 5000 annotations...
Processed 10000 annotations...
Processed 15000 annotations...
Processed 20000 annotations...
Processed 25000 annotations...
Processed 30000 annotations...
Processed 35000 annotations...
Processed 40000 annotations...
Processed 45000 annotations...
Processed 50000 annotations...
Processed 55000 annotations...
Processed 60000 annotations...
Processed 65000 annotations...
Processed 70000 annotations...
Processed 75000 annotations...
Processed 80000 annotations...
Processed 85000 annotations...
Processed 90000 annotations...
Processed 95000 annotations...
Processed 100000 annotations...
Processed 105000 annotations...
Processed 110000 annotations...
Processed 115000 annotations...
Processed 120000 annotations...
Processed 125000 annotations...
Processed 130000 annotations...
Processed 135000 annotations...
Processed 140000 annotations...
Processed 145000 annotations...
Pro

In [5]:
# CELL 4: Prune VALIDATION split (FAST & SAFE)

import os
import json
import glob
import shutil

# --- paths ---
VAL_IMG_DIR = os.path.join(BASE_DIR, "validation", "validation", "image")
VAL_ANNO_DIR = os.path.join(BASE_DIR, "validation", "validation", "annos")

OUT_VAL_IMG_DIR = os.path.join(BASE_DIR, "DeepFashion2_pruned", "validation", "images")
OUT_VAL_ANNO_DIR = os.path.join(BASE_DIR, "DeepFashion2_pruned", "validation", "annos")

os.makedirs(OUT_VAL_IMG_DIR, exist_ok=True)
os.makedirs(OUT_VAL_ANNO_DIR, exist_ok=True)

print("Pruning VALIDATION split...")
print("Top-5 categories used:", top5)

kept_images = 0
total_images = 0

for ann_file in glob.glob(os.path.join(VAL_ANNO_DIR, "*.json")):
    total_images += 1

    img_id = os.path.basename(ann_file).replace(".json", "")
    img_path = os.path.join(VAL_IMG_DIR, img_id + ".jpg")

    if not os.path.exists(img_path):
        continue

    with open(ann_file, "r") as f:
        data = json.load(f)

    items = []
    for k, v in data.items():
        if k.startswith("item") and v["category_id"] in top5:
            items.append({
                "category_id": v["category_id"],
                "bbox": v["bounding_box"],
                "segmentation": v["segmentation"]
            })

    # skip images with no top-5 items
    if not items:
        continue

    # copy image
    shutil.copy(img_path, os.path.join(OUT_VAL_IMG_DIR, img_id + ".jpg"))

    # save pruned annotation
    with open(os.path.join(OUT_VAL_ANNO_DIR, img_id + ".json"), "w") as f:
        json.dump({"items": items}, f)

    kept_images += 1

print("Total VALIDATION images scanned:", total_images)
print("VALIDATION images kept after pruning:", kept_images)

Pruning VALIDATION split...
Top-5 categories used: [1, 8, 7, 2, 9]
Total VALIDATION images scanned: 32153
VALIDATION images kept after pruning: 23741


In [10]:
# CELL 5: Prepare TEST split (images only)

import os
import glob
import shutil

TEST_IMG_DIR = os.path.join(BASE_DIR, "test", "test", "image")
OUT_TEST_IMG_DIR = os.path.join(BASE_DIR, "DeepFashion2_pruned", "test", "images")

os.makedirs(OUT_TEST_IMG_DIR, exist_ok=True)

print("Copying TEST images...")

count = 0
for img_path in glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg")):
    img_name = os.path.basename(img_path)
    out_path = os.path.join(OUT_TEST_IMG_DIR, img_name)

    if not os.path.exists(out_path):
        shutil.copy(img_path, out_path)
        count += 1

print("Total TEST images copied:", count)

Copying TEST images...
Total TEST images copied: 62629
